In [0]:
%run "./00_config"


In [0]:
%run "./01_core_functions"


## Run the pipeline

For every active source in `source_control`:
1. list every file currently in its folder
2. check the log table for each file's most recent result
3. load anything new or previously failed
4. log every outcome

Re-running this notebook is safe: files whose last logged attempt was
`SUCCESS` are skipped, only new or failed files get processed

In [0]:
run_pipeline()


In [0]:
ls = dbutils.fs.ls("s3://cittabase-s3-test/Databricks-poc/json_files/")

In [0]:
ls

In [0]:
from pyspark.sql.types import StructType, ArrayType

In [0]:
# Read the JSON file
df = (
    spark.read
         .option("multiline", "true")
         .json("/Volumes/iics/casemix/history/json_test/extracted/carereport_hsar_20250305_162803/carereport_hsar_5110097798747137794_20250224_20250305_20250305_162803.json")
)

In [0]:
# Recursive function to extract schema paths
def extract_schema(schema, parent=""):
    fields = []

    for field in schema.fields:
        name = f"{parent}.{field.name}" if parent else field.name
        dtype = field.dataType

        if isinstance(dtype, StructType):
            fields.append(name)
            fields.extend(extract_schema(dtype, name))

        elif isinstance(dtype, ArrayType):
            fields.append(f"{name}[]")

            if isinstance(dtype.elementType, StructType):
                fields.extend(extract_schema(dtype.elementType, f"{name}[]"))

        else:
            fields.append(name)

    return fields

schema_list = extract_schema(df.schema)


In [0]:
schema_list

In [0]:

def extract_schema_with_type(schema, parent=""):
    fields = []

    for field in schema.fields:
        name = f"{parent}.{field.name}" if parent else field.name
        dtype = field.dataType

        fields.append((name, dtype.simpleString()))

        if isinstance(dtype, StructType):
            fields.extend(extract_schema_with_type(dtype, name))

        elif isinstance(dtype, ArrayType):
            if isinstance(dtype.elementType, StructType):
                fields.extend(
                    extract_schema_with_type(dtype.elementType, f"{name}[]")
                )

    return fields

schema_info = extract_schema_with_type(df.schema)


In [0]:
schema_info

In [0]:
for name, dtype in schema_info:
    print(f"{name} : {dtype}")